# Week 44, Day 3: Open addressing

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w044d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
slots = [None] * 7
for key in [10, 3, 17, 24, 5]:
    i = key % 7
    probes = 1
    while slots[i] is not None:
        i = (i + 1) % 7
        probes += 1
    slots[i] = key
    print(key, "-> slot", i, "after", probes, "probes")
print(slots)

In [ ]:
EMPTY = None

class OpenMap:
    def __init__(self, capacity=8):
        self._keys = [EMPTY] * capacity
        self._values = [EMPTY] * capacity
        self._size = 0

    def _probe(self, key):
        """The slot holding key, or the first empty slot on its path."""
        m = len(self._keys)
        i = hash(key) % m
        while self._keys[i] is not EMPTY and self._keys[i] != key:
            i = (i + 1) % m
        return i

    def put(self, key, value):
        i = self._probe(key)
        if self._keys[i] is EMPTY:
            self._keys[i] = key
            self._size += 1
        self._values[i] = value

    def get(self, key, default=None):
        i = self._probe(key)
        return default if self._keys[i] is EMPTY else self._values[i]

m = OpenMap()
for k, v in [(10, "a"), (18, "b"), (26, "c"), (3, "d"), (18, "B")]:
    m.put(k, v)
print(m._keys, m.get(18), m.get(26), m.get(34, "missing"), m._size)

In [ ]:
EMPTY, DELETED = None, object()          # object() makes a unique marker nothing else equals

def probe_find(keys, key):
    m, i = len(keys), hash(key) % len(keys)
    while keys[i] is not EMPTY:
        if keys[i] is not DELETED and keys[i] == key:
            return i
        i = (i + 1) % m
    return -1

keys = [EMPTY] * 8
for k in [10, 18, 26]:                      # all want slot 2
    i = k % 8
    while keys[i] is not EMPTY:
        i = (i + 1) % 8
    keys[i] = k
print([k if k is not DELETED else "†" for k in keys])

keys[probe_find(keys, 18)] = EMPTY          # the wrong way
print("emptied:", probe_find(keys, 26))
keys[3] = 18
keys[probe_find(keys, 18)] = DELETED        # the right way
print("tombstone:", probe_find(keys, 26), [k if k is not DELETED else "†" for k in keys])

In [ ]:
def longest_run(keys, m, step_for):
    slots = [None] * m
    for k in keys:
        i, s = k % m, step_for(k)
        while slots[i] is not None:
            i = (i + s) % m
        slots[i] = k
    run = best = 0
    for x in slots + slots:                  # twice round, to count runs that wrap
        run = run + 1 if x is not None else 0
        best = max(best, run)
    return min(best, m)

import random
rng = random.Random(1)
keys = rng.sample(range(10 ** 6), 700)
m = 1009                                      # a prime
print("linear :", longest_run(keys, m, lambda k: 1))
print("double :", longest_run(keys, m, lambda k: 1 + k % (m - 1)))

> **Key idea.** Open addressing stores every key directly in the table. A key whose home slot is taken probes along a fixed sequence (linear probing: the next slot, wrapping round); lookups follow the same path and stop at an empty slot. Deleting must leave a tombstone, which lookups probe past and inserts may reuse; rebuild the table to clear them. Probes grow sharply with the load factor, so resize early (around 0.5 to 0.7, counting tombstones), reinserting live keys only. Linear probing causes primary clustering; quadratic probing and double hashing spread keys out. Open addressing saves memory and is memory-friendly; chaining handles deletes and high load more simply.

### Exercise: Linear probing by hand

Write `insert(slots, key)` for a list of slots (`None` for empty) and a whole-number key whose home slot is `key % len(slots)`: put the key in the first free slot from its home (wrapping round), and return `(index, probes)`, where `probes` counts every slot looked at, including the one used. A full table raises `OverflowError`. Then set `layout` to the slots after inserting `[12, 22, 32, 5, 15, 9]` into 10 empty slots.

In [ ]:
def insert(slots, key):
    ___

layout = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_insert():\n    \"probing\"\n    s = [None] * 5\n    assert insert(s, 7) == (2, 1) and insert(s, 12) == (3, 2) and insert(s, 4) == (4, 1) and insert(s, 9) == (0, 2)\n    assert s == [9, None, 7, 12, 4]\n    s = [1, 2]\n    try:\n        insert(s, 3)\n        assert False\n    except OverflowError:\n        pass\n    assert layout == [None, None, 12, 22, 32, 5, 15, None, None, 9]")

### Exercise: An open-addressed map

Write `OpenMap(capacity=8)` with linear probing over two parallel lists, `_keys` and `_values`, using `None` for an empty slot: `put` (updating an existing key), `get(key, default=None)`, `__contains__` and `__len__`. After adding a new key, if `len / capacity` is more than 0.5, double the capacity and reinsert every key. Keep `capacity` as a read-only property. Keys are never `None`.

In [ ]:
class OpenMap:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_map():\n    \"a map\"\n    m = OpenMap(4)\n    for k, v in [(\"a\", 1), (\"b\", 2), (\"a\", 3)]:\n        m.put(k, v)\n    assert len(m) == 2 and m.get(\"a\") == 3 and m.get(\"z\", 0) == 0 and \"b\" in m and \"z\" not in m and m.capacity == 4\n    caps = []\n    for i in range(20):\n        m.put(i, i * 2)\n        caps.append(m.capacity)\n    assert all(m.get(i) == i * 2 for i in range(20)) and len(m) == 22 and m.capacity == 64\n    assert len(m._keys) == len(m._values) == m.capacity\n    assert all(c >= 2 * n for c, n in zip(caps, range(3, 23)))\ndef test_no_lists_of_lists():\n    \"no chains\"\n    m = OpenMap()\n    for i in range(1000):\n        m.put(i, i)\n    assert not any(isinstance(k, list) for k in m._keys) and m.get(999) == 999")

### Exercise: Deleting with tombstones

Add `delete(key)` to the starter's map: replace the key with the `DELETED` marker (and its value with `None`), and return the value; a missing key raises `KeyError(key)`. Then fix the other methods so that lookups **probe past** tombstones, and `put` of a **new** key reuses the **first** tombstone on its path (but only after checking the key isn't further along). Count tombstones in `self._tombstones`.

In [ ]:
EMPTY, DELETED = None, object()

class OpenMap:
    def __init__(self, capacity=16):
        self._keys = [EMPTY] * capacity
        self._values = [None] * capacity
        self._size = 0
        self._tombstones = 0

    def _slot(self, key):
        m = len(self._keys)
        i = hash(key) % m
        while self._keys[i] is not EMPTY and self._keys[i] != key:
            i = (i + 1) % m
        return i

    def put(self, key, value):
        i = self._slot(key)
        if self._keys[i] is EMPTY:
            self._keys[i] = key
            self._size += 1
        self._values[i] = value

    def get(self, key, default=None):
        i = self._slot(key)
        return default if self._keys[i] is EMPTY else self._values[i]

    def __len__(self):
        return self._size

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_delete():\n    \"tombstones\"\n    m = OpenMap(16)\n    for k in [1, 17, 33]:            # all want slot 1\n        m.put(k, str(k))\n    assert m.delete(17) == \"17\" and m.get(33) == \"33\" and m.get(17) is None and len(m) == 2 and m._tombstones == 1\n    assert m._keys[2] is DELETED\n    m.put(49, \"49\")\n    assert m._keys[2] == 49 and m._tombstones == 0, \"reuse the first tombstone\"\n    m.delete(1)\n    m.put(33, \"again\")\n    assert m.get(33) == \"again\" and len(m) == 2 and sum(1 for k in m._keys if k == 33) == 1, \"don't put a second copy into the tombstone\"\n    try:\n        m.delete(1)\n        assert False\n    except KeyError:\n        pass\ndef test_random():\n    \"against a dict\"\n    import random\n    rng = random.Random(1)\n    m, ref = OpenMap(512), {}\n    for _ in range(3000):\n        k = rng.randint(0, 120)\n        if rng.random() < 0.5:\n            m.put(k, k)\n            ref[k] = k\n        elif k in ref:\n            assert m.delete(k) == ref.pop(k)\n        assert m.get(k) == ref.get(k) and len(m) == len(ref)")

### Exercise: Probes against load

Write `average_probes(load, m=1009, seed=1)`: insert `int(load * m)` random distinct keys (use `random.Random(seed).sample(range(10**7), n)`) into `m` slots with linear probing, then return the average number of probes needed to **find** each key (rounded to 2 places), where finding a key takes as many probes as inserting it did. Then set `table` to a list of `average_probes` for loads 0.25, 0.5, 0.75 and 0.9.

In [ ]:
import random

def average_probes(load, m=1009, seed=1):
    ___

table = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_probes():\n    \"the experiment\"\n    assert average_probes(0.0) == 0\n    assert table[0] < table[1] < table[2] < table[3] and table[0] < 1.4 and table[3] > 3\n    assert abs(table[1] - 1.5) < 0.4, \"close to Knuth's 1.5 at half full\"")

### Exercise: Double hashing

Write `probe_sequence(key, m)` for double hashing: the list of all `m` slot indexes the key would try, starting at `key % m` and stepping by `1 + key % (m - 1)` each time (wrapping with `% m`). Then `visits_all(m)`: true if, for every key from 0 to `m * 3`, the sequence visits **every** slot exactly once. (It's true when `m` is prime.)

In [ ]:
def probe_sequence(key, m):
    ___

def visits_all(m):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_double():\n    \"double hashing\"\n    assert probe_sequence(10, 7) == [3, 1, 6, 4, 2, 0, 5]\n    assert probe_sequence(3, 5) == [3, 2, 1, 0, 4]\n    assert visits_all(7) and visits_all(11) and visits_all(13) and not visits_all(8) and not visits_all(9)")

### Bug hunt: Stopped by a tombstone

This lookup stops at a tombstone as if it were empty, so a key placed after the deleted one can't be found: this prints `None`. Make lookups probe past tombstones, so it prints `c`.

In [ ]:
EMPTY, DELETED = None, object()
keys = [EMPTY] * 8
values = [None] * 8
for k, v in [(2, "a"), (10, "b"), (18, "c")]:        # all want slot 2
    i = k % 8
    while keys[i] is not EMPTY:
        i = (i + 1) % 8
    keys[i], values[i] = k, v
keys[3] = DELETED                                     # 10 was deleted

def get(key):
    i = key % 8
    while keys[i] is not EMPTY and keys[i] is not DELETED:
        if keys[i] == key:
            return values[i]
        i = (i + 1) % 8
    return None

print(get(18))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"probe past tombstones\"\n    expect_output(\"c\")")

### Bug hunt: Deleted by emptying

This `delete` empties the slot, which breaks the probe path of every key placed after it: deleting 10 makes 18 unreachable, and this prints `False`. Leave a tombstone instead (and make `contains` probe past it), so it prints `True`.

In [ ]:
EMPTY = None
keys = [EMPTY] * 8

def add(key):
    i = key % 8
    while keys[i] is not EMPTY:
        i = (i + 1) % 8
    keys[i] = key

def contains(key):
    i = key % 8
    while keys[i] is not EMPTY:
        if keys[i] == key:
            return True
        i = (i + 1) % 8
    return False

def delete(key):
    i = key % 8
    while keys[i] != key:
        i = (i + 1) % 8
    keys[i] = EMPTY

for k in [2, 10, 18]:
    add(k)
delete(10)
print(contains(18))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a tombstone\"\n    expect_output(\"True\")\n    assert not contains(10)")

### Bug hunt: Off the end

This probe moves to the next slot without wrapping round, so a key whose home is near the end of the table runs off it and crashes with `IndexError`. Fix it so this prints `[13, None, None, None, None, None, 6]`.

In [ ]:
slots = [None] * 7

def insert(key):
    i = key % 7
    while slots[i] is not None:
        i = i + 1
    slots[i] = key

insert(6)
insert(13)
print(slots)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"wrap round\"\n    expect_output(\"[13, None, None, None, None, None, 6]\")")

### Puzzle: Finding a key

Arrange the lines into a lookup for an open-addressed table with tombstones. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code. Tables use linear probing and hash(n) == n.

p = ___   # the slot 15 ends up in, inserted into 8 empty slots after 7 and 23 (in that order)
q = ___   # how many probes a search for a missing key takes in a completely empty table
r = ___   # True or False: an open-addressed table can hold more keys than it has slots
s = ___   # what a lookup does at a tombstone: "stop" or "continue"
t = ___   # what a lookup does at an empty slot: "stop" or "continue"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 1 and q == 1 and r is False, \"p, q and r: 7 goes in 7, 23 in 0, then 15 in 1\"\n    assert s == \"continue\" and t == \"stop\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a complete open-addressed map

Write `OpenDict` with linear probing over parallel `_keys` and `_values` lists, tombstones for deletion, and these rules: start with 8 slots; after adding a new key, if `(live keys + tombstones) / capacity` is more than 0.6, rebuild: to **double** the capacity if live keys alone are more than 0.3 of it, otherwise at the **same** capacity (just clearing the tombstones). Give it `d[key]` (with `KeyError`), `d[key] = value`, `del d[key]`, `in`, `len`, iteration over keys, `items()`, `get`, and read-only `capacity` and `tombstones` properties. A key must never appear twice.

In [ ]:
EMPTY, DELETED = None, object()

class OpenDict:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_basic():\n    \"basics\"\n    d = OpenDict()\n    d[\"a\"], d[\"b\"] = 1, 2\n    d[\"a\"] = 10\n    assert d[\"a\"] == 10 and len(d) == 2 and \"b\" in d and sorted(d) == [\"a\", \"b\"] and sorted(d.items()) == [(\"a\", 10), (\"b\", 2)]\n    del d[\"a\"]\n    assert \"a\" not in d and d.get(\"a\", \"x\") == \"x\" and d.tombstones == 1\n    for bad in [lambda: d[\"a\"], lambda: d.__delitem__(\"a\")]:\n        try:\n            bad()\n            assert False\n        except KeyError:\n            pass\ndef test_churn():\n    \"many adds and deletes keep the table small\"\n    d = OpenDict()\n    for i in range(10_000):\n        d[i] = i\n        del d[i]\n    assert len(d) == 0 and d.capacity == 8 and d.tombstones <= 5\ndef test_random():\n    \"against a dict\"\n    rng = random.Random(9)\n    d, ref = OpenDict(), {}\n    for _ in range(8000):\n        k = rng.randint(0, 400)\n        r = rng.random()\n        if r < 0.5:\n            d[k] = ref[k] = rng.randint(0, 9)\n        elif r < 0.8 and k in ref:\n            del d[k], ref[k]\n        assert d.get(k) == ref.get(k) and len(d) == len(ref)\n    assert sorted(d.items()) == sorted(ref.items())\n    live = [k for k in d._keys if k is not EMPTY and k is not DELETED]\n    assert len(live) == len(set(live)) == len(ref)\n    assert (len(ref) + d.tombstones) / d.capacity <= 0.6")

## Warm-up

10, 17 and 24 all want slot 3. The first gets it; the others step along to the next free slots, 4 and 5; then 5 wants slot 5 too, finds it taken, and steps on to 6. No lists anywhere: collisions are settled by **moving along**.

## Probing

With **linear probing**, a key whose home slot `h = hash(key) % m` is taken tries `h + 1`, `h + 2`, ... (wrapping round with `% m`) until it finds an empty slot. To **find** a key, follow exactly the same path: check `h`, then `h + 1`, ..., stopping when you find the key (success) or reach an **empty** slot (the key isn't there, because if it were, it would have been put here or earlier).

Two keys and two values sit in **parallel arrays** at the same index. (A single array of pairs would work too.) The loop in `_probe` always ends as long as the table is never completely full, so a resize must happen well before that.

## Deleting needs tombstones

Deleting is where open addressing gets subtle. Suppose 10, 18 and 26 all wanted slot 2 of 8, and sit in slots 2, 3 and 4. Delete 18 by emptying slot 3, and then look for 26: the search starts at slot 2 (holding 10), moves to slot 3, finds it **empty**, and stops. 26 is still in slot 4, but nobody can reach it.

The fix is a **tombstone**: a special marker that means "something was here". Deleting replaces the key with a tombstone.

- **Lookups** treat a tombstone as occupied: keep probing past it.
- **Inserts** may reuse a tombstone's slot, but only after checking the key isn't already further along the path.

`DELETED = object()` creates a marker that is equal only to itself, so it can never be mistaken for a real key. Compare with `is`, not `==`.

Tombstones have a cost: they make probe paths longer, and they never go away on their own. So a map counts its tombstones and, when there are too many, **rebuilds** the table (reinserting only live keys), which clears them all.

## Load and resizing

Open addressing is much more sensitive to the load factor than chaining. As the table fills, the runs of occupied slots get longer, and the number of probes grows sharply. Donald Knuth worked out the expected number of probes for linear probing with load factor α:

| α | probes to find a key that's there | probes to discover a key isn't there |
|---|---|---|
| 0.25 | about 1.2 | about 1.4 |
| 0.5 | 1.5 | 2.5 |
| 0.75 | 2.5 | 8.5 |
| 0.9 | 5.5 | 50.5 |

(The formulas are ½(1 + 1/(1 − α)) and ½(1 + 1/(1 − α)²), from *The Art of Computer Programming*, volume 3.) So open-addressed tables usually resize at α around 0.5 to 0.7, counting tombstones as used slots, since they lengthen paths just as keys do. Resizing reinserts every **live** key into a bigger empty table: tombstones are simply left behind.

## Clusters

Linear probing has a weakness: **primary clustering**. Once a few neighbouring slots are full, any key whose home is anywhere in that run lands at its end, making it longer, which makes it even more likely to catch the next key. Runs merge into long clusters.

Two common fixes change the probe sequence:

- **Quadratic probing**: try `h`, `h + 1`, `h + 4`, `h + 9`, ... (adding squares), so keys with nearby homes spread out quickly.
- **Double hashing**: use a **second** hash function to pick the step size: try `h`, `h + s`, `h + 2s`, ..., where `s = hash2(key)`. Keys with the same home slot follow different paths. The step must never be 0, and should share no factor with `m` (a prime `m` makes every step work), so the sequence visits every slot.

Linear probing is still widely used, because walking to the very next slot is fast for the computer's memory (the same reason Python lists beat linked lists, Week 42). Python's dict uses its own probe sequence that mixes in more and more of the hash's bits (tomorrow).

## Chaining or open addressing?

| | Chaining | Open addressing |
|---|---|---|
| collisions | share a list in the bucket | probe for another slot |
| load factor | can exceed 1 (chains just get longer) | must stay below 1, usually under 0.7 |
| deleting | easy: remove from the chain | needs tombstones |
| memory | a list per bucket, plus pairs | just the arrays; no extra objects |
| speed in practice | following pointers | neighbouring slots: friendly to memory |

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a table of 8 slots, insert 2, 10, 18 and 26 with linear probing, delete 10, then search for 26 twice: once as if deleting emptied the slot, once with a tombstone. Tomorrow: how Python's own dictionary works, with open addressing, a clever probe sequence, and insertion order kept for free.